# 02 — Building the recruitment matching engine

> **Applied Labs** · **Domain**: AG · **Difficulty**: Advanced · **Reading time**: ~40 min

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/labs/08_recruitment_matching/02_build.ipynb)

**Prerequisites**:
- [01_architecture.ipynb](./01_architecture.ipynb)

**What you will learn**:
- How to build realistic sample data with diverse candidates and multiple job descriptions
- Structured JD requirement extraction with OpenAI
- Resume parsing with explicit and inferred skill extraction
- Full semantic matching engine across all JD–candidate combinations
- Bias detection and fairness analysis on scoring results
- End-to-end demo with ranked scorecards and interview question generation

In [ ]:
# @title Setup — Run this cell first
!pip install -q "openai>=1.0.0" "sentence-transformers>=2.2.0" "pandas>=2.0.0" "numpy>=1.24.0"

import os, json, re, textwrap, time
from dataclasses import dataclass, field, asdict
from typing import Optional, Dict, List, Tuple, Set
from collections import defaultdict
import numpy as np
import pandas as pd

from openai import OpenAI
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ── Clients ──
client: Optional[OpenAI] = None
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    client = OpenAI(api_key=api_key)
    print("OpenAI client initialised ✓")
else:
    print("⚠ OPENAI_API_KEY not set — LLM calls will use mock responses")

encoder = SentenceTransformer("all-MiniLM-L6-v2")
print("Sentence encoder loaded ✓")
np.random.seed(42)
print("Setup complete ✓")

## Section 1 — Sample data

We create three job descriptions targeting different roles and ten candidate
resumes with deliberately diverse backgrounds, education, terminology, and
career paths. This diversity is essential for testing semantic matching and
bias detection.

In [ ]:
# ── Core data classes ──

@dataclass
class Requirement:
    """A single extracted requirement from a JD."""
    id: str
    text: str
    category: str   # technical | experience | education | soft-skill
    priority: str   # required | preferred
    min_years: float = 0.0

@dataclass
class ParsedJD:
    """Structured job description."""
    title: str
    level: str
    requirements: List[Requirement] = field(default_factory=list)
    responsibilities: List[str] = field(default_factory=list)

@dataclass
class ExtractedSkill:
    """Skill extracted from a resume."""
    name: str
    source: str      # explicit | inferred
    evidence: str
    years: float = 0.0

@dataclass
class ResumeProfile:
    """Structured candidate profile."""
    name: str
    skills: List[ExtractedSkill] = field(default_factory=list)
    total_years: float = 0.0
    education: str = ""
    achievements: List[str] = field(default_factory=list)
    domains: List[str] = field(default_factory=list)
    background_group: str = ""  # for fairness analysis (simulated)

@dataclass
class MatchResult:
    """Match result for one requirement vs one candidate."""
    requirement_id: str
    requirement_text: str
    match_level: str   # exact | semantic | partial | none
    score: float
    candidate_skill: str
    evidence: str

@dataclass
class CriterionScore:
    """Score for one rubric criterion."""
    criterion: str
    weight: float
    score: int
    evidence: str
    reasoning: str

@dataclass
class Scorecard:
    """Complete evaluation scorecard."""
    candidate_name: str
    job_title: str
    criterion_scores: List[CriterionScore] = field(default_factory=list)
    overall_score: float = 0.0
    recommendation: str = ""
    match_results: List[MatchResult] = field(default_factory=list)

# ── Three job descriptions ──
JOB_DESCRIPTIONS: Dict[str, str] = {
    "senior_engineer": """Senior Software Engineer — Backend Platform

We are looking for a Senior Software Engineer to design and build our
core platform services.

Requirements:
- 5+ years of backend development experience
- Strong proficiency in Python or Go
- Experience with distributed systems and microservices
- Proficiency in SQL and database design (PostgreSQL preferred)
- Experience with cloud platforms (AWS or GCP)
- Understanding of CI/CD pipelines and DevOps practices

Preferred:
- Experience with Kubernetes and container orchestration
- Knowledge of event-driven architectures (Kafka, RabbitMQ)
- Contributions to open-source projects
- Experience mentoring junior engineers""",

    "data_scientist": """Senior Data Scientist — ML Platform

Join our ML team to build and deploy production machine learning systems.

Requirements:
- 4+ years of machine learning experience
- Strong Python skills with PyTorch or TensorFlow
- Experience with ML model deployment and monitoring
- Proficiency in SQL and data pipeline tools
- Strong statistical analysis and experimental design skills
- Bachelor's degree in CS, Statistics, or related field

Preferred:
- Experience with large language models and NLP
- Knowledge of MLOps tools (MLflow, Kubeflow)
- Published research or conference presentations
- Experience with A/B testing at scale""",

    "product_manager": """Senior Product Manager — AI Products

Lead the product strategy for our AI-powered features.

Requirements:
- 5+ years of product management experience
- Experience shipping AI/ML-powered products
- Strong data analysis and metrics-driven decision making
- Excellent cross-functional communication and stakeholder management
- Understanding of machine learning concepts and limitations
- Bachelor's degree required; MBA or technical degree preferred

Preferred:
- Experience with enterprise B2B products
- Background in user research and design thinking
- Experience managing a product team of 3+
- Technical background (CS degree or engineering experience)""",
}

# ── Ten diverse candidate resumes ──
RESUMES: List[Dict[str, str]] = [
    {"name": "Alice Chen", "group": "A", "text": """Alice Chen — Senior Software Engineer
7 years of experience in backend development and distributed systems.
Experience: Lead Engineer at CloudScale (2020-present) — designed microservices
architecture handling 50K RPS. Built event-driven pipelines with Kafka.
Previously at MegaCorp (2017-2020) — Python, Go, PostgreSQL, Redis.
Skills: Python, Go, PostgreSQL, Redis, Kafka, Kubernetes, Docker, AWS, CI/CD
Education: MS Computer Science, UC Berkeley
Achievements: Open-source contributor (2K+ GitHub stars), mentor to 5 junior devs"""},

    {"name": "Bob Martinez", "group": "A", "text": """Bob Martinez — ML Engineer
5 years in applied machine learning and data engineering.
Experience: ML Engineer at DataFlow (2021-present) — deployed 12 production
models using PyTorch, MLflow. Reduced training costs by 40%.
Previously at AnalyticsInc (2019-2021) — built data pipelines, A/B testing.
Skills: Python, PyTorch, TensorFlow, SQL, MLflow, Spark, Docker
Education: MS Statistics, Stanford
Achievements: Published at NeurIPS 2022, led ML reading group"""},

    {"name": "Carol Okafor", "group": "B", "text": """Carol Okafor — Computational Physicist
8 years of research in computational physics and numerical methods.
Experience: Research Fellow at CERN (2019-present) — developed Monte Carlo
simulations, statistical analysis of particle collision data, pattern
recognition algorithms processing petabytes of data.
Previously at National Lab (2016-2019) — numerical computing, R, MATLAB.
Skills: R, MATLAB, C++, statistical modelling, data analysis, HPC, Linux
Education: PhD Physics, Oxford University
Achievements: 12 publications, developed novel analysis framework used by 200+ researchers"""},

    {"name": "David Park", "group": "A", "text": """David Park — Full Stack Developer
4 years of web development with some backend experience.
Experience: Developer at WebStudio (2021-present) — React, Node.js, MongoDB.
Built customer dashboard serving 10K users.
Previously freelance developer (2020-2021) — WordPress, PHP, JavaScript.
Skills: JavaScript, React, Node.js, MongoDB, HTML, CSS, PHP
Education: BS Information Technology, State University
Achievements: Built 3 client projects, active on Stack Overflow"""},

    {"name": "Elena Volkov", "group": "B", "text": """Elena Volkov — Ingénieur en apprentissage automatique
6 years in artificial intelligence and natural language processing.
Experience: AI Engineer at EuroTech (2020-present) — built NLP pipelines
for document classification, sentiment analysis, entity extraction.
Deployed transformer models serving 5M requests/day.
Previously Research Engineer at AI Lab Paris (2018-2020) — deep learning,
computer vision, reinforcement learning.
Skills: Python, PyTorch, Transformers, NLP, Docker, Kubernetes, SQL, GCP
Education: Diplôme d'ingénieur (equivalent MS), École Polytechnique
Achievements: 3 patents filed, deployed 8 production ML systems"""},

    {"name": "Frank Liu", "group": "A", "text": """Frank Liu — Senior Product Manager
6 years in product management, 3 years in AI products.
Experience: Senior PM at TechGiant (2021-present) — led AI-powered search
and recommendation products serving 100M users. Managed team of 4 PMs.
Previously PM at StartupAI (2019-2021) — shipped 3 ML features from 0 to 1.
Before that: Software Engineer (2017-2019) — Python, data analysis.
Skills: Product strategy, roadmapping, A/B testing, data analysis, SQL, Jira
Education: MBA, Wharton; BS Computer Science, Georgia Tech
Achievements: Grew AI product revenue by 300%, led cross-functional team of 20"""},

    {"name": "Grace Nakamura", "group": "B", "text": """Grace Nakamura — Data Analyst → Aspiring Data Scientist
3 years in data analysis, transitioning to machine learning.
Experience: Senior Data Analyst at RetailCo (2021-present) — SQL, Tableau,
Python for analysis. Built customer segmentation model (self-directed).
Previously Business Analyst at ConsultingFirm (2020-2021) — Excel, SQL.
Completed ML specialisation on Coursera (Andrew Ng). Built 5 ML projects.
Skills: Python, SQL, Tableau, Excel, pandas, scikit-learn, basic PyTorch
Education: BS Mathematics, University of Michigan
Achievements: Automated 3 reporting workflows, presented at internal data summit"""},

    {"name": "Hassan Al-Rashid", "group": "B", "text": """Hassan Al-Rashid — Systems Architect
10 years in infrastructure and platform engineering.
Experience: Principal Engineer at ScaleUp (2019-present) — architected
multi-region Kubernetes platform serving 500+ microservices. Led migration
from monolith. Reduced infrastructure costs by 45%.
Previously at Enterprise Corp (2014-2019) — Java, Spring, Oracle, Linux.
Skills: Kubernetes, Docker, Terraform, AWS, GCP, Go, Python, PostgreSQL, Kafka
Education: BS Computer Engineering, MIT
Achievements: Scaled platform to 99.99% uptime, mentor to 10+ engineers"""},

    {"name": "Iris Thompson", "group": "A", "text": """Iris Thompson — Product Designer → Product Manager
4 years in product roles, design background.
Experience: Product Manager at DesignTech (2022-present) — user research,
design thinking, shipped 2 AI-assisted design features.
Previously UX Designer at Creative Agency (2020-2022) — user interviews,
prototyping, usability testing. Data-informed design decisions.
Skills: User research, design thinking, Figma, data analysis, basic SQL, Jira
Education: BFA Design, Rhode Island School of Design; PM certification
Achievements: Improved user retention by 25% through redesign"""},

    {"name": "James Wright", "group": "A", "text": """James Wright — Junior Developer (Career Changer)
1 year of professional development experience. Former teacher.
Experience: Junior Developer at CodeStart (2023-present) — Python, Django,
basic SQL. Building internal tools.
Previously: High school mathematics teacher (2015-2023) — data analysis
for student performance, curriculum development.
Completed bootcamp (Flatiron School) — Python, JavaScript, SQL, React.
Skills: Python, JavaScript, SQL, Django, React, HTML, CSS
Education: BA Mathematics Education, NYU; Flatiron School bootcamp
Achievements: Built student analytics dashboard used by 50 teachers"""},
]

print("=" * 60)
print("  SAMPLE DATA SUMMARY")
print("=" * 60)
print(f"  Job descriptions : {len(JOB_DESCRIPTIONS)}")
for key, jd in JOB_DESCRIPTIONS.items():
    title = jd.strip().split("\n")[0]
    print(f"    • {title}")
print(f"\n  Candidate resumes: {len(RESUMES)}")
for r in RESUMES:
    group_tag = f"[{r['group']}]"
    first_line = r["text"].strip().split("\n")[0]
    print(f"    {group_tag} {first_line}")
print(f"\n✓ {len(JOB_DESCRIPTIONS)} JDs × {len(RESUMES)} candidates "
      f"= {len(JOB_DESCRIPTIONS) * len(RESUMES)} evaluations")

## Section 2 — JD requirement extractor

We use OpenAI structured output to parse each job description into categorised,
prioritised requirements. The extractor distinguishes must-have from nice-to-have
and assigns categories (technical, experience, education, soft-skill).

In [ ]:
JD_PARSE_PROMPT = """You are an expert technical recruiter. Parse this job description
into structured requirements.

JOB DESCRIPTION:
{jd_text}

For each requirement determine:
- category: technical | experience | education | soft-skill
- priority: required | preferred
- min_years: minimum years (0 if not specified)

Respond in this exact JSON format:
{{
  "title": "<job title>",
  "level": "<junior|mid|senior|lead|principal>",
  "requirements": [
    {{"id": "R1", "text": "<requirement>", "category": "<cat>", "priority": "<pri>", "min_years": <float>}}
  ],
  "responsibilities": ["<resp1>", "<resp2>"]
}}"""

def parse_jd(jd_text: str) -> ParsedJD:
    """Parse a job description into structured requirements."""
    if client:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": JD_PARSE_PROMPT.format(jd_text=jd_text)}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        data = json.loads(resp.choices[0].message.content)
    else:
        # ── Mock: rule-based extraction ──
        reqs: List[Dict] = []
        for line in jd_text.split("\n"):
            line = line.strip("- •*").strip()
            if len(line) < 8:
                continue
            cat, pri, yrs = "technical", "required", 0.0
            if any(w in line.lower() for w in ["prefer", "nice", "bonus"]):
                pri = "preferred"
            if any(w in line.lower() for w in ["degree", "bachelor", "master", "mba"]):
                cat = "education"
            elif any(w in line.lower() for w in ["communicat", "collaborat", "stakeholder", "cross-functional"]):
                cat = "soft-skill"
            yr_match = re.search(r"(\d+)\+?\s*(?:years?|yrs?)", line.lower())
            if yr_match:
                yrs = float(yr_match.group(1))
                if cat == "technical":
                    cat = "experience"
            if any(c.isalpha() for c in line):
                reqs.append({"id": f"R{len(reqs)+1}", "text": line,
                             "category": cat, "priority": pri, "min_years": yrs})
        title = jd_text.strip().split("\n")[0]
        data = {"title": title, "level": "senior",
                "requirements": reqs[:12], "responsibilities": []}

    return ParsedJD(
        title=data.get("title", ""),
        level=data.get("level", ""),
        requirements=[Requirement(**r) for r in data.get("requirements", [])],
        responsibilities=data.get("responsibilities", []),
    )

# ── Parse all 3 JDs ──
parsed_jds: Dict[str, ParsedJD] = {}
for key, jd_text in JOB_DESCRIPTIONS.items():
    parsed_jds[key] = parse_jd(jd_text)

print("=" * 60)
print("  JD REQUIREMENT EXTRACTION")
print("=" * 60)
for key, pjd in parsed_jds.items():
    req_count = len([r for r in pjd.requirements if r.priority == "required"])
    pref_count = len([r for r in pjd.requirements if r.priority == "preferred"])
    print(f"\n  {pjd.title}")
    print(f"  Level: {pjd.level} | Required: {req_count} | Preferred: {pref_count}")
    for req in pjd.requirements:
        tag = "REQ" if req.priority == "required" else "PRF"
        yrs = f" ({req.min_years:.0f}+ yrs)" if req.min_years > 0 else ""
        print(f"    [{tag}] {req.id}: {req.text[:60]}{yrs}")
print(f"\n✓ Parsed {len(parsed_jds)} job descriptions")

## Section 3 — Resume parser and skill extractor

The resume parser extracts explicit skills (mentioned directly) and inferred
skills (derived from experience context). This is critical for matching
non-traditional candidates whose terminology differs from job descriptions.

In [ ]:
RESUME_PARSE_PROMPT = """Parse this resume into a structured profile.

RESUME:
{resume_text}

Extract ALL skills — both explicitly mentioned and inferred from experience.
For inferred skills, cite the evidence.

Respond in JSON:
{{
  "name": "<name>",
  "total_years": <float>,
  "education": "<degree and field>",
  "skills": [{{"name": "<skill>", "source": "explicit|inferred",
               "evidence": "<text>", "years": <float>}}],
  "achievements": ["<a1>"],
  "domains": ["<d1>"]
}}"""

# ── Skill inference rules for mock mode ──
SKILL_INFERENCES: Dict[str, List[str]] = {
    "microservices": ["Distributed Systems", "API Design"],
    "deployed": ["DevOps", "CI/CD"],
    "pipeline": ["Data Engineering", "ETL"],
    "led": ["Leadership", "Team Management"],
    "published": ["Research", "Technical Writing"],
    "mentor": ["Mentoring", "Leadership"],
    "scaled": ["System Design", "Performance Engineering"],
    "migration": ["System Design", "Risk Management"],
    "a/b test": ["Experimentation", "Statistical Analysis"],
    "user research": ["UX Research", "Design Thinking"],
    "cross-functional": ["Stakeholder Management", "Communication"],
    "transformer": ["NLP", "Deep Learning"],
    "monte carlo": ["Statistical Modelling", "Simulation"],
    "pattern recognition": ["Machine Learning", "Statistical Analysis"],
}

KNOWN_SKILLS: List[str] = [
    "python", "java", "go", "c++", "javascript", "r", "matlab", "sql",
    "pytorch", "tensorflow", "scikit-learn", "pandas", "numpy", "spark",
    "kubernetes", "docker", "terraform", "aws", "gcp", "azure",
    "kafka", "redis", "postgresql", "mongodb", "react", "node.js",
    "mlflow", "kubeflow", "ci/cd", "linux", "git", "tableau", "excel",
    "jira", "figma", "django", "spring", "nlp", "deep learning",
    "machine learning", "data analysis", "design thinking",
]

def parse_resume(resume_text: str, group: str = "") -> ResumeProfile:
    """Parse a resume into a structured candidate profile."""
    if client:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": RESUME_PARSE_PROMPT.format(
                resume_text=resume_text)}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        data = json.loads(resp.choices[0].message.content)
    else:
        # ── Mock: keyword + inference extraction ──
        skills: List[Dict] = []
        text_lower = resume_text.lower()
        for skill in KNOWN_SKILLS:
            if skill in text_lower:
                skills.append({"name": skill.title(), "source": "explicit",
                               "evidence": f"Found '{skill}' in resume", "years": 2.0})
        for trigger, inferred_skills in SKILL_INFERENCES.items():
            if trigger in text_lower:
                for inf in inferred_skills:
                    if not any(s["name"].lower() == inf.lower() for s in skills):
                        skills.append({"name": inf, "source": "inferred",
                                       "evidence": f"Inferred from '{trigger}'", "years": 0.0})
        # Extract years
        yr_match = re.search(r"(\d+)\s*years?", text_lower)
        years = float(yr_match.group(1)) if yr_match else 3.0
        # Extract education
        edu = "Unknown"
        for deg in ["PhD", "MS", "MBA", "BS", "BA", "BFA", "Diplôme"]:
            if deg.lower() in text_lower or deg in resume_text:
                edu_match = re.search(rf"{deg}[^\n]*", resume_text, re.IGNORECASE)
                if edu_match:
                    edu = edu_match.group(0).strip()
                    break
        # Extract name
        name = resume_text.strip().split("\n")[0].split("—")[0].strip()
        data = {"name": name, "total_years": years, "education": edu,
                "skills": skills, "achievements": [], "domains": []}

    return ResumeProfile(
        name=data.get("name", "Unknown"),
        total_years=data.get("total_years", 0.0),
        education=data.get("education", ""),
        skills=[ExtractedSkill(**s) for s in data.get("skills", [])],
        achievements=data.get("achievements", []),
        domains=data.get("domains", []),
        background_group=group,
    )

# ── Parse all 10 resumes ──
profiles: List[ResumeProfile] = []
for r in RESUMES:
    profiles.append(parse_resume(r["text"], r["group"]))

print("=" * 60)
print("  RESUME PARSING — 10 CANDIDATES")
print("=" * 60)
for p in profiles:
    explicit = sum(1 for s in p.skills if s.source == "explicit")
    inferred = sum(1 for s in p.skills if s.source == "inferred")
    print(f"  {p.name:<20} {p.total_years:>4.0f} yrs | "
          f"{explicit:>2} explicit + {inferred:>2} inferred = {len(p.skills):>2} skills | "
          f"{p.education[:30]}")
print(f"\n✓ Parsed {len(profiles)} candidate profiles")

## Section 4 — Semantic matching engine

The matching engine evaluates every JD × resume combination. For each
requirement, it finds the best matching skill using embedding similarity
and classifies the match level. This produces a full match matrix.

In [ ]:
def semantic_match(
    requirements: List[Requirement],
    profile: ResumeProfile,
    threshold_semantic: float = 0.65,
    threshold_partial: float = 0.35,
) -> List[MatchResult]:
    """Match each requirement against candidate skills using embeddings."""
    if not profile.skills or not requirements:
        return [MatchResult(r.id, r.text, "none", 0.0, "", "")
                for r in requirements]

    req_texts = [r.text for r in requirements]
    skill_texts = [s.name for s in profile.skills]

    req_embs = encoder.encode(req_texts)
    skill_embs = encoder.encode(skill_texts)
    sim = cosine_similarity(req_embs, skill_embs)

    results: List[MatchResult] = []
    for i, req in enumerate(requirements):
        best_j = int(np.argmax(sim[i]))
        best_score = float(sim[i][best_j])
        best_skill = profile.skills[best_j]

        if (req.text.lower().strip() in best_skill.name.lower() or
                best_skill.name.lower() in req.text.lower()):
            level, score = "exact", 1.0
        elif best_score >= threshold_semantic:
            level, score = "semantic", best_score
        elif best_score >= threshold_partial:
            level, score = "partial", best_score
        else:
            level, score = "none", 0.0

        results.append(MatchResult(
            requirement_id=req.id, requirement_text=req.text,
            match_level=level, score=round(score, 3),
            candidate_skill=best_skill.name, evidence=best_skill.evidence,
        ))
    return results

def evaluate_candidate(
    profile: ResumeProfile,
    matches: List[MatchResult],
    job_title: str,
) -> Scorecard:
    """Score candidate using structured rubric."""
    strong = [m for m in matches if m.match_level in ("exact", "semantic")]
    tech_score = min(5, max(1, int(len(strong) / max(len(matches), 1) * 5) + 1))

    yrs = profile.total_years
    exp_score = (1 if yrs < 1 else 2 if yrs < 3 else 3 if yrs < 5
                 else 4 if yrs < 8 else 5)

    edu_lower = profile.education.lower()
    edu_score = (5 if "phd" in edu_lower else 4 if any(d in edu_lower for d in ["master", "ms ", "mba", "diplôme"])
                 else 3 if any(d in edu_lower for d in ["bachelor", "bs ", "ba "]) else 2)

    culture_score = min(5, max(2, len(profile.achievements) + 1))
    growth_score = min(5, max(2, len(profile.domains) + 1))

    criteria = [
        CriterionScore("Technical Skills", 0.35, tech_score,
                       f"{len(strong)}/{len(matches)} reqs matched", "Semantic matching"),
        CriterionScore("Experience", 0.25, exp_score,
                       f"{profile.total_years} years", "Year-based scoring"),
        CriterionScore("Education", 0.15, edu_score,
                       profile.education[:50], "Degree mapping"),
        CriterionScore("Culture", 0.15, culture_score,
                       f"{len(profile.achievements)} achievements", "Achievement count"),
        CriterionScore("Growth", 0.10, growth_score,
                       f"{len(profile.domains)} domains", "Domain breadth"),
    ]
    overall = round(sum(c.weight * c.score for c in criteria), 2)
    rec = ("strong-yes" if overall >= 4.0 else "yes" if overall >= 3.2
           else "maybe" if overall >= 2.5 else "no")

    return Scorecard(profile.name, job_title, criteria, overall, rec, matches)

# ── Run all JD × candidate combinations ──
all_scorecards: Dict[str, List[Scorecard]] = {}

for jd_key, pjd in parsed_jds.items():
    scorecards: List[Scorecard] = []
    for profile in profiles:
        matches = semantic_match(pjd.requirements, profile)
        sc = evaluate_candidate(profile, matches, pjd.title)
        scorecards.append(sc)
    all_scorecards[jd_key] = sorted(scorecards, key=lambda s: s.overall_score, reverse=True)

# ── Match matrix ──
print("=" * 70)
print("  MATCH MATRIX — Overall Scores (JD × Candidate)")
print("=" * 70)
header = f"  {'Candidate':<20}" + "".join(f"{pjd.title[:18]:>20}" for pjd in parsed_jds.values())
print(header)
print("  " + "-" * (20 + 20 * len(parsed_jds)))
for i, profile in enumerate(profiles):
    row = f"  {profile.name:<20}"
    for jd_key in parsed_jds:
        sc = next(s for s in all_scorecards[jd_key] if s.candidate_name == profile.name)
        row += f"{sc.overall_score:>18.2f}  "
    print(row)
print(f"\n✓ Generated {sum(len(v) for v in all_scorecards.values())} scorecards")

## Section 5 — Bias detection and fairness

We analyse scoring patterns across simulated demographic groups to check for
disparate impact. We also examine whether the scoring system systematically
undervalues non-traditional backgrounds.

In [ ]:
def fairness_analysis(
    scorecards: List[Scorecard],
    profiles: List[ResumeProfile],
    threshold: float = 3.0,
) -> Dict[str, object]:
    """Analyse scoring fairness across demographic groups."""
    # Map candidate names to groups
    name_to_group: Dict[str, str] = {p.name: p.background_group for p in profiles}

    group_scores: Dict[str, List[float]] = defaultdict(list)
    for sc in scorecards:
        group = name_to_group.get(sc.candidate_name, "?")
        group_scores[group].append(sc.overall_score)

    results: Dict[str, object] = {}
    groups = sorted(group_scores.keys())
    if len(groups) >= 2:
        a, b = groups[0], groups[1]
        scores_a, scores_b = group_scores[a], group_scores[b]
        mean_a, mean_b = np.mean(scores_a), np.mean(scores_b)
        pass_a = sum(1 for s in scores_a if s >= threshold) / len(scores_a)
        pass_b = sum(1 for s in scores_b if s >= threshold) / len(scores_b)
        air = pass_b / pass_a if pass_a > 0 else 1.0

        results["group_means"] = {a: round(mean_a, 3), b: round(mean_b, 3)}
        results["pass_rates"] = {a: round(pass_a, 3), b: round(pass_b, 3)}
        results["adverse_impact_ratio"] = round(air, 3)
        results["four_fifths_pass"] = air >= 0.8
        results["mean_gap"] = round(abs(mean_a - mean_b), 3)

    return results

# ── Run fairness analysis for each JD ──
print("=" * 60)
print("  FAIRNESS ANALYSIS — BY JOB DESCRIPTION")
print("=" * 60)
for jd_key, scorecards in all_scorecards.items():
    pjd = parsed_jds[jd_key]
    report = fairness_analysis(scorecards, profiles)

    print(f"\n  {pjd.title}")
    print(f"  " + "-" * 50)
    if "group_means" in report:
        for g, m in report["group_means"].items():
            print(f"    Group {g} mean score  : {m:.2f}")
        print(f"    Mean gap            : {report['mean_gap']:.3f}")
        for g, r in report["pass_rates"].items():
            print(f"    Group {g} pass rate   : {r:.1%}")
        air = report["adverse_impact_ratio"]
        status = "✓ PASS" if report["four_fifths_pass"] else "✗ FAIL"
        print(f"    Adverse impact ratio: {air:.3f}  [{status}]")

# ── Qualification vs score analysis ──
print("\n" + "-" * 60)
print("  QUALIFICATION vs SCORE — CHECKING FOR BIAS")
print("-" * 60)
jd_key = "data_scientist"
for sc in all_scorecards[jd_key]:
    p = next(p for p in profiles if p.name == sc.candidate_name)
    bar = "█" * int(sc.overall_score) + "░" * (5 - int(sc.overall_score))
    print(f"  [{p.background_group}] {sc.candidate_name:<20} {bar} {sc.overall_score:.2f}  "
          f"({p.total_years:.0f} yrs, {len(p.skills)} skills)")
print("\n✓ Fairness analysis complete — review groups for disparate impact.")

## Section 6 — End-to-end demo

For each job, we rank candidates with full scorecards and generate tailored
interview questions for the top candidates based on their profiles and
identified gaps.

In [ ]:
INTERVIEW_PROMPT = """Generate 3 tailored interview questions for this candidate
applying to {job_title}.

CANDIDATE: {candidate_name}
OVERALL SCORE: {score}/5.0
STRENGTHS: {strengths}
GAPS: {gaps}

Generate questions that:
1. Probe the identified gaps constructively
2. Let the candidate demonstrate strengths
3. Assess culture fit

Respond in JSON: {{"questions": ["<q1>", "<q2>", "<q3>"]}}"""

def generate_interview_questions(
    scorecard: Scorecard,
    profile: ResumeProfile,
) -> List[str]:
    """Generate tailored interview questions for a candidate."""
    strengths = ", ".join(m.candidate_skill for m in scorecard.match_results
                         if m.match_level in ("exact", "semantic"))[:200]
    gaps = ", ".join(m.requirement_text for m in scorecard.match_results
                     if m.match_level == "none")[:200]

    if client:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": INTERVIEW_PROMPT.format(
                job_title=scorecard.job_title,
                candidate_name=scorecard.candidate_name,
                score=scorecard.overall_score,
                strengths=strengths or "N/A",
                gaps=gaps or "N/A",
            )}],
            temperature=0.3,
            response_format={"type": "json_object"},
        )
        data = json.loads(resp.choices[0].message.content)
        return data.get("questions", [])
    else:
        # ── Mock questions ──
        qs: List[str] = []
        if gaps:
            qs.append(f"Your background doesn't explicitly mention {gaps.split(',')[0].strip()[:40]}. "
                      "Can you describe any related experience?")
        qs.append(f"Tell us about a project where you used {strengths.split(',')[0].strip()[:30]} "
                  "to solve a challenging problem.")
        qs.append("Describe a time you collaborated across teams with different "
                  "technical backgrounds. What was the outcome?")
        return qs[:3]

# ── End-to-end demo ──
for jd_key, scorecards in all_scorecards.items():
    pjd = parsed_jds[jd_key]
    print("╔" + "═" * 62 + "╗")
    print(f"║  JOB: {pjd.title:<54}║")
    print("╠" + "═" * 62 + "╣")

    for rank, sc in enumerate(scorecards[:5], 1):
        profile = next(p for p in profiles if p.name == sc.candidate_name)
        rec_emoji = {"strong-yes": "🟢", "yes": "🟡", "maybe": "🟠", "no": "🔴"}.get(sc.recommendation, "⚪")
        print(f"║  #{rank} {sc.candidate_name:<18} {sc.overall_score:.2f}/5.0  "
              f"{rec_emoji} {sc.recommendation:<18}║")
        for cs in sc.criterion_scores:
            bar = "█" * cs.score + "░" * (5 - cs.score)
            print(f"║    {cs.criterion:<16} {bar} {cs.score}/5 ({cs.evidence[:25]}){' ' * 3}║")

        # Top 3 get interview questions
        if rank <= 3:
            questions = generate_interview_questions(sc, profile)
            print(f"║  Interview questions:{' ' * 41}║")
            for qi, q in enumerate(questions, 1):
                wrapped = textwrap.shorten(q, width=56)
                print(f"║    Q{qi}: {wrapped:<53}║")
        print("║" + " " * 62 + "║")

    print("╚" + "═" * 62 + "╝")
    print()

print("✓ End-to-end recruitment matching demo complete.")

In [ ]:
# ── Visualise match matrix heatmap ──
import matplotlib.pyplot as plt

jd_names = [parsed_jds[k].title[:20] for k in parsed_jds]
cand_names = [p.name for p in profiles]
matrix = np.zeros((len(profiles), len(parsed_jds)))
for j, jd_key in enumerate(parsed_jds):
    for i, profile in enumerate(profiles):
        sc = next(s for s in all_scorecards[jd_key] if s.candidate_name == profile.name)
        matrix[i][j] = sc.overall_score

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(matrix, cmap="RdYlGn", aspect="auto", vmin=1, vmax=5)
ax.set_xticks(range(len(jd_names)))
ax.set_xticklabels(jd_names, rotation=30, ha="right")
ax.set_yticks(range(len(cand_names)))
ax.set_yticklabels(cand_names)
for i in range(len(cand_names)):
    for j in range(len(jd_names)):
        ax.text(j, i, f"{matrix[i][j]:.1f}", ha="center", va="center", fontsize=8)
ax.set_title("Match Matrix — Overall Scores (JD × Candidate)")
fig.colorbar(im, ax=ax, label="Score (1-5)")
plt.tight_layout()
plt.show()
print("✓ Match matrix heatmap rendered")

## Exercises

1. **Add candidate diversity** — Add 5 more resumes representing different career
   paths: military veteran, academic researcher transitioning to industry, self-taught
   developer, returning parent, and international candidate. Run the full pipeline and
   check whether scoring is equitable.

2. **Improve skill inference** — Extend `SKILL_INFERENCES` with 15 more trigger
   phrases. Measure how many additional skills are captured across all 10 resumes.
   Calculate the improvement in match coverage.

3. **Custom rubric weights** — Create three rubric configurations: one prioritising
   technical skills (0.50 weight), one prioritising experience (0.40 weight), and
   one prioritising culture fit (0.35 weight). Compare how rankings change across
   all three JDs.

## Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | Realistic sample data with diverse backgrounds is essential for testing fairness |
| 2 | JD parsing separates must-have from nice-to-have — critical for accurate matching |
| 3 | Inferred skills from context significantly improve matching for non-traditional candidates |
| 4 | The 3 JDs × 10 candidates matrix reveals which candidates are versatile across roles |
| 5 | Bias detection at every stage catches disparate impact before it reaches hiring decisions |
| 6 | Tailored interview questions help hiring managers probe gaps constructively |
| 7 | End-to-end scorecards with evidence make hiring decisions auditable and defensible |

## What's Next

In **03 — Evaluate**, we measure matching accuracy against expert rankings,
test rubric consistency, run comprehensive fairness analysis, and assess
explanation quality using LLM-as-judge.

---

## References

1. Fuller, J. et al., *Hidden Workers: Untapped Talent*, Harvard Business School, 2021 — <https://www.hbs.edu/>
2. EEOC, *Uniform Guidelines on Employee Selection Procedures*, 1978 — <https://www.eeoc.gov/>
3. Raghavan, M. et al., *Mitigating Bias in Algorithmic Hiring: Evaluating Claims and Practices*, FAT* 2020.
4. Sentence-Transformers library — <https://www.sbert.net/>